In [212]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

In [213]:
airbnb=pd.read_excel('airbnb.xlsx')
airbnb.head()

,id,log_price,property_type,room_type,amenities,accommodates,bathrooms,bed_type,cancellation_policy,cleaning_fee,...,latitude,longitude,name,neighbourhood,number_of_reviews,review_scores_rating,thumbnail_url,zipcode,bedrooms,beds
0,6901257,5.010635,Apartment,Entire home/apt,"{""Wireless Internet"",""Air conditioning"",Kitche...",3,1.0,Real Bed,strict,True,...,40.696524,-73.991617,Beautiful brownstone 1-bedroom,Brooklyn Heights,2,100.0,https://a0.muscache.com/im/pictures/6d7cbbf7-c...,11201,1.0,1.0
1,6304928,5.129899,Apartment,Entire home/apt,"{""Wireless Internet"",""Air conditioning"",Kitche...",7,1.0,Real Bed,strict,True,...,40.766115,-73.989040,Superb 3BR Apt Located Near Times Square,Hell's Kitchen,6,93.0,https://a0.muscache.com/im/pictures/348a55fe-4...,10019,3.0,3.0
2,7919400,4.976734,Apartment,Entire home/apt,"{TV,""Cable TV"",""Wireless Internet"",""Air condit...",5,1.0,Real Bed,moderate,True,...,40.808110,-73.943756,The Garden Oasis,Harlem,10,92.0,https://a0.muscache.com/im/pictures/6fae5362-9...,10027,1.0,3.0
3,13418779,6.620073,House,Entire home/apt,"{TV,""Cable TV"",Internet,""Wireless Internet"",Ki...",4,1.0,Real Bed,flexible,True,...,37.772004,-122.431619,Beautiful Flat in the Heart of SF!,Lower Haight,0,NaN,https://a0.muscache.com/im/pictures/72208dad-9...,94117,2.0,2.0
4,3808709,4.744932,Apartment,Entire home/apt,"{TV,Internet,""Wireless Internet"",""Air conditio...",2,1.0,Real Bed,moderate,True,...,38.925627,-77.034596,Great studio in midtown DC,Columbia Heights,4,40.0,NaN,20009,0.0,1.0


In [214]:
#리뷰개수 11 이상인 자료 추출, 재산 타입 변수 변환
df1=airbnb[airbnb['number_of_reviews']>=11]
df1.loc[(df1['property_type']!='Apartment')&(df1['property_type']!='House'), 'property_type']='Other'

In [215]:
# 침대 유형 변수 변환, 침대 유형 unique 활용하여 확인함
df1.loc[df1['bed_type'] == 'Real Bed', 'bed_type'] = 'Bed'
df1.loc[df1['bed_type'] != 'Bed', 'bed_type'] = 'Other'

In [216]:
#어메니티 개수, 설명서 길이 측정하여 변수 변환
df1 = df1.assign(
    amenities_count = df1['amenities'].str.strip('{}').str.split(',').str.len()
)
df1 = df1.assign(
    description_count = df1['description'].str.len()
)
df1.head()

,id,log_price,property_type,room_type,amenities,accommodates,bathrooms,bed_type,cancellation_policy,cleaning_fee,...,name,neighbourhood,number_of_reviews,review_scores_rating,thumbnail_url,zipcode,bedrooms,beds,amenities_count,description_count
6,11825529,4.418841,Apartment,Entire home/apt,"{TV,Internet,""Wireless Internet"",""Air conditio...",3,1.0,Bed,moderate,True,...,Beach Town Studio and Parking!!!11h,NaN,15,97.0,https://a0.muscache.com/im/pictures/4c920c60-4...,90292,1.0,1.0,21,1000
8,180792,4.787492,House,Private room,"{TV,""Cable TV"",""Wireless Internet"",""Pets live ...",2,1.0,Bed,moderate,True,...,Cozy Garden Studio - Private Entry,Richmond District,159,99.0,https://a0.muscache.com/im/pictures/0ed6c128-7...,94121,1.0,1.0,21,1000
10,5578513,4.605170,Apartment,Private room,"{Internet,""Wireless Internet"",""Air conditionin...",2,1.0,Bed,strict,True,...,Large East Village Bedroom To Let!,Alphabet City,82,93.0,https://a0.muscache.com/im/pictures/21726900/1...,10009,1.0,1.0,15,1000
11,17423675,5.010635,House,Entire home/apt,"{TV,""Cable TV"",Internet,""Wireless Internet"",Ki...",4,1.5,Bed,strict,True,...,Sand Section Beach Bungalow,Hermosa Beach,29,97.0,NaN,90254,2.0,2.0,22,1000
13,2658946,5.298317,Apartment,Entire home/apt,"{TV,""Cable TV"",Internet,""Wireless Internet"",""A...",6,1.5,Bed,strict,True,...,Charming 2 bdrm in trendy U/14th streets w/par...,U Street Corridor,13,89.0,NaN,20009,2.0,3.0,25,1000


In [217]:
# 도시별 위도, 경도의 평균(중심점) 계산
city_center = df1.groupby('city')[['latitude', 'longitude']].mean().reset_index()
city_center.columns = ['city', 'center_lat', 'center_lon']
df1 = df1.merge(city_center, on='city', how='left')

# 도시 중심으로부터의 거리 계산
import numpy as np
df1['dist_from_center'] = np.log(np.sqrt(
    (df1['latitude'] - df1['center_lat'])**2 + 
    (df1['longitude'] - df1['center_lon'])**2
))

In [218]:
#원가격 구하고 도시별 평균 가격 구하기
df1['original_price']=np.exp(df1['log_price'])
df1['city_mean_price'] = df1.groupby('city')['original_price'].transform('mean')

In [219]:
#가격비 계산
df1['price_ratio']=(df1['original_price']/df1['city_mean_price'])*100
df1['log_price_ratio']=np.log(df1['price_ratio'])

In [220]:
#불필요한 열 삭제
df1=df1.drop(['id', 'name', 'description', 'thumbnail_url', 'zipcode','first_review','last_review','amenities','neighbourhood'], axis=1)
df1.head()

,log_price,property_type,room_type,accommodates,bathrooms,bed_type,cancellation_policy,cleaning_fee,city,host_has_profile_pic,...,beds,amenities_count,description_count,center_lat,center_lon,dist_from_center,original_price,city_mean_price,price_ratio,log_price_ratio
0,4.418841,Apartment,Entire home/apt,3,1.0,Bed,moderate,True,LA,t,...,1.0,21,1000,34.053350,-118.341298,-1.953976,83.0,140.122352,59.233947,4.081495
1,4.787492,House,Private room,2,1.0,Bed,moderate,True,SF,t,...,1.0,21,1000,37.763572,-122.433416,-2.660412,120.0,189.819051,63.218101,4.146591
2,4.605170,Apartment,Private room,2,1.0,Bed,strict,True,NYC,t,...,1.0,15,1000,40.730808,-73.953791,-3.477791,100.0,142.604285,70.124120,4.250267
3,5.010635,House,Entire home/apt,4,1.5,Bed,strict,True,LA,t,...,2.0,22,1000,34.053350,-118.341298,-1.671297,150.0,140.122352,107.049302,4.673289
4,5.298317,Apartment,Entire home/apt,6,1.5,Bed,strict,True,DC,t,...,3.0,25,1000,38.911664,-77.018008,-4.173297,200.0,137.181602,145.792146,4.982182


In [221]:
#리뷰스코어 공백인 경우 스코어가 없는 것으로 판단
df1['review_scores_rating']=df1['review_scores_rating'].fillna(0)
# 프로필 사진 여부 공백인 경우 없는 것으로 판단
df1['host_has_profile_pic'] = df1['host_has_profile_pic'].fillna('f')
# 호스트 증명 여부 공백인 경우 없는 것으로 판단
df1['host_identity_verified'] = df1['host_identity_verified'].fillna('f')
#응답률 공백을 응답률 0으로 보기 어려워 중앙값으로 결측치 채움
response_median = df1['host_response_rate'].median()
df1['host_response_rate'] = df1['host_response_rate'].fillna(response_median)

In [222]:
#언제부터 지어졌는지 판단하기 위해서 날짜화, 기준일로부터의 일수로 변수 변환
df1['host_since'] = pd.to_datetime(df1['host_since'])
reference_date = pd.to_datetime('2026-04-09') 
df1['host_since2'] = (reference_date - df1['host_since']).dt.days

In [223]:
#결측치가 미미하여 결측치있는 행 제거, 결측치를 임의로 채울 경우 왜곡될 우려
df1 =df1.dropna(subset=['bedrooms', 'beds', 'bathrooms','host_since2'])

In [224]:
#불필요한 열 삭제
df1=df1.drop(['host_since','latitude','longitude','center_lat','center_lon','city_mean_price'], axis=1)

In [225]:
#범주형 자료 더미변수화
cols = ['property_type','room_type','bed_type','cancellation_policy','cleaning_fee', 'city','host_has_profile_pic','host_identity_verified','instant_bookable']
airbnb_dummies=pd.get_dummies(df1,columns=cols,drop_first=True)
#단위차이가 커 표준화
airbnb_dummies = (airbnb_dummies - airbnb_dummies.mean()) / airbnb_dummies.std()

In [226]:
# Private room이면서 수용 인원이 많을 때의 효과, 설명이 길 때 재산 유형이 집인 경우의 효과 측정
airbnb_dummies['acc_x_private'] = airbnb_dummies['accommodates'] * airbnb_dummies['room_type_Private room']
airbnb_dummies['desc_x_house'] = airbnb_dummies['description_count'] * airbnb_dummies['property_type_House']

In [227]:
#다중공선성 확인
from statsmodels.stats.outliers_influence import variance_inflation_factor
vif_df = pd.DataFrame()
vif_df["feature"] = X.columns
vif_df["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
print(vif_df.sort_values("VIF", ascending=False))

                                feature       VIF
1                          accommodates  5.348148
6                                  beds  3.604895
21                              city_LA  2.970127
5                              bedrooms  2.791468
22                             city_NYC  2.617897
24                        acc_x_private  2.518351
16           cancellation_policy_strict  2.467681
15         cancellation_policy_moderate  2.352706
12               room_type_Private room  2.338933
0                                 const  1.828406
9                      dist_from_center  1.598039
2                             bathrooms  1.534356
19                         city_Chicago  1.400632
20                              city_DC  1.398510
10                  property_type_House  1.383628
7                       amenities_count  1.197166
13                room_type_Shared room  1.169664
4                  review_scores_rating  1.129889
11                  property_type_Other  1.122455


In [228]:
#회귀분석 진행, 후진제거법으로 유의하지 않은 설명변수 제거, 제거 후 추가도 해보았으나 달라지지 않았음, 
#다중공선성이 10이 넘지 않으나 수용인원과 베드 개수는 연관성이 커보여 'beds'를 제외했더니 더 높은 설명력
x=airbnb_dummies.drop(columns=['log_price_ratio','log_price','original_price','price_ratio','host_has_profile_pic_t','city_SF','host_identity_verified_t','cleaning_fee_True','host_since2','number_of_reviews'])
y=airbnb_dummies['log_price_ratio']
x = x.astype(float)
X = sm.add_constant(x)
airbnb.reg = sm.OLS(y,X).fit()
airbnb.reg.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:        log_price_ratio   R-squared:                       0.625
Model:                            OLS   Adj. R-squared:                  0.625
Method:                 Least Squares   F-statistic:                     1918.
Date:                Thu, 09 Apr 2026   Prob (F-statistic):               0.00
Time:                        04:39:15   Log-Likelihood:                -26747.
No. Observations:               28802   AIC:                         5.355e+04
Df Residuals:                   28776   BIC:                         5.376e+04
Df Model:                          25                                         
Covariance Type:            nonrobust                                         
=======================================================================================================
                                          coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------------------
const                                   0.0292      0.005      5.971      0.000       0.020       0.039
accommodates                            0.2538      0.008     30.398      0.000       0.237       0.270
bathrooms                               0.1061      0.004     23.724      0.000       0.097       0.115
host_response_rate                     -0.0221      0.004     -6.005      0.000      -0.029      -0.015
review_scores_rating                    0.1053      0.004     27.428      0.000       0.098       0.113
bedrooms                                0.2377      0.006     39.405      0.000       0.226       0.250
beds                                   -0.0825      0.007    -12.034      0.000      -0.096      -0.069
amenities_count                         0.0408      0.004     10.330      0.000       0.033       0.049
description_count                       0.0079      0.004      2.099      0.036       0.001       0.015
dist_from_center                       -0.1224      0.005    -26.819      0.000      -0.131      -0.113
property_type_House                    -0.0229      0.004     -5.398      0.000      -0.031      -0.015
property_type_Other                     0.0118      0.004      3.081      0.002       0.004       0.019
room_type_Private room                 -0.4297      0.006    -77.826      0.000      -0.441      -0.419
room_type_Shared room                  -0.2588      0.004    -66.288      0.000      -0.266      -0.251
bed_type_Other                         -0.0106      0.004     -2.850      0.004      -0.018      -0.003
cancellation_policy_moderate            0.0163      0.006      2.943      0.003       0.005       0.027
cancellation_policy_strict              0.0535      0.006      9.433      0.000       0.042       0.065
cancellation_policy_super_strict_30     0.0184      0.004      5.053      0.000       0.011       0.025
cancellation_policy_super_strict_60     0.0129      0.004      3.578      0.000       0.006       0.020
city_Chicago                           -0.0223      0.004     -5.216      0.000      -0.031      -0.014
city_DC                                -0.0286      0.004     -6.699      0.000      -0.037      -0.020
city_LA                                 0.0248      0.006      3.989      0.000       0.013       0.037
city_NYC                                0.0835      0.006     14.287      0.000       0.072       0.095
instant_bookable_t                     -0.0365      0.004     -9.840      0.000      -0.044      -0.029
acc_x_private                           0.0675      0.007      9.211      0.000       0.053       0.082
desc_x_house                            0.0272      0.004      7.296      0.000       0.020       0.035
===================================================================

In [229]:
#설명변수에 의해 가격비의 62.5%가 설명되었음, 나머지 37.5%는 이 회귀분석으로 설명되지 못하는 부분
#기울기 해석(설명변수 표준화 시킴)
#bedrooms= 0.2377. 방 개수가 1 표준편차 증가하면, 가격비는 평균 23.77% 증가함
#accommodates = 0.2538. 수용 인원이 1 표준편차 증가하면, 가격비는 평균 25.38% 증가함
#dist_from_center = -0.1224. 도심 중심으로부터의 거리(로그 취함)가 1 표준편차 멀어질수록, 가격비는 평균 12.24% 감소함
#review_scores_rating = 0.1053. 리뷰 점수가 1 표준편차 증가하면, 가격비는 평균 10.53% 증가함
#room_type_Private room = -0.4297. 숙소 형태가 'Entire home/apt'에서 'private(개인실)'로 바뀌면, 가격비는 평균 42.97% 낮게 형성됨
#acc_x_private = 0.0675. (상호작용) 숙소가 개인실이면서 수용 인원이 1 표준편차 증가할 때, 가격비는 평균 6.75% 상승함
#desc_x_house = 0.02732. (상호작용) 숙소 형태가 주택(House)이면서 소개문 길이가 1 표준편차 길어지면, 가격비는 평균 2.72%  증가함.

#방 개수, 수용 인원, 중심으로부터의 거리, 리뷰 점수, 숙소 형태가 에어비앤비 로그가격비에 강력한 영향을 지닌다.

In [230]:
#테스트, 훈련 변수로 분할
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=1234)

In [231]:
#선형회귀분석
from sklearn.linear_model import LinearRegression
r_linear = LinearRegression(fit_intercept = True)
r_linear.fit(X_train ,y_train)

LinearRegression()

In [232]:
#의사경정나무
from sklearn.tree import DecisionTreeRegressor
r_tree = DecisionTreeRegressor(min_impurity_decrease=0.01,random_state=0)
r_tree.fit(X_train, y_train)

DecisionTreeRegressor(min_impurity_decrease=0.01, random_state=0)

In [233]:
#랜덤포레스트
from sklearn.ensemble import RandomForestRegressor
r_rf = RandomForestRegressor(random_state=0)
r_rf.fit(X_train,y_train)

RandomForestRegressor(random_state=0)

In [234]:
#mae 계산, Random Forest가 가장 낮게 나옴,오차가 적음->설명력 좋음
from sklearn.metrics import mean_absolute_error as mae
print('Linear Regression:',np.round(mae(y_test,r_linear.predict(X_test)),2))
print('Regression Tree  :',np.round(mae(y_test,r_tree.predict(X_test)),2))
print('Random Forest    :',np.round(mae(y_test,r_rf.predict(X_test)),2))

Linear Regression: 0.47
Regression Tree  : 0.51
Random Forest    : 0.43


In [235]:
#mape 계산, Random Forest가 가장 낮게 나옴->설명력 좋음
from sklearn.metrics import mean_absolute_percentage_error as mape
print('Linear Regression:',np.round(mape(y_test,r_linear.predict(X_test)),2))
print('Regression Tree  :',np.round(mape(y_test,r_tree.predict(X_test)),2))
print('Random Forest    :',np.round(mape(y_test,r_rf.predict(X_test)),2))

Linear Regression: 2.21
Regression Tree  : 2.23
Random Forest    : 2.11


In [236]:
#mse 계산, Random Forest가 가장 낮게 나옴->설명력 좋음
from sklearn.metrics import mean_squared_error as mse
print('Linear Regression:',np.round(mse(y_test,r_linear.predict(X_test)),2))
print('Regression Tree  :',np.round(mse(y_test,r_tree.predict(X_test)),2))
print('Random Forest    :',np.round(mse(y_test,r_rf.predict(X_test)),2))

Linear Regression: 0.38
Regression Tree  : 0.43
Random Forest    : 0.33
